In [1]:
import pandas as pd
import numpy as np
import os
import requests
import time
import random

# Configuration
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
REAL_CATS_DATA_DIR = os.path.join(PROJECT_ROOT, 'Real_Cats_data')
PATH_BENIGN = os.path.join(REAL_CATS_DATA_DIR, 'BB.tsv')
PATH_CRIMINAL = os.path.join(REAL_CATS_DATA_DIR, 'CB.tsv')

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Directory: {REAL_CATS_DATA_DIR}")

Project Root: d:\Projects\final_project
Data Directory: d:\Projects\final_project\Real_Cats_data


In [2]:
def fetch_wallet_transactions(address, first_time=None, last_time=None, max_retries=3):
    """
    Fetch all transactions for a single wallet address from mempool.space API.
    Returns a list of edge dictionaries (source, target, weight, timestamp, direction).
    """
    edges_list = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Parse time window
    try:
        start_ts = pd.to_datetime(first_time).timestamp() if first_time else 0
        end_ts = pd.to_datetime(last_time).timestamp() if last_time else 9999999999
    except:
        start_ts, end_ts = 0, 9999999999
    
    retries = 0
    while retries <= max_retries:
        try:
            url = f"https://mempool.space/api/address/{address}/txs"
            r = requests.get(url, headers=headers, timeout=15)
            
            if r.status_code == 200:
                txs = r.json()
                
                for tx in txs:
                    # Check if transaction is confirmed
                    if not tx.get('status', {}).get('confirmed'):
                        continue
                        
                    tx_time = tx['status']['block_time']
                    
                    # Check time window
                    if tx_time < start_ts or tx_time > end_ts:
                        continue
                    
                    # Check if wallet is sender (appears in inputs)
                    is_sender = False
                    for inp in tx.get('vin', []):
                        if inp.get('prevout', {}).get('scriptpubkey_address') == address:
                            is_sender = True
                            break
                    
                    # If sender, create edges to all recipients
                    if is_sender:
                        for out in tx.get('vout', []):
                            recipient = out.get('scriptpubkey_address')
                            amount = out.get('value', 0)  # Satoshis
                            
                            if recipient and recipient != address:
                                edges_list.append({
                                    'source': address,
                                    'target': recipient,
                                    'weight': amount,
                                    'timestamp': tx_time,
                                    'direction': 'outgoing',
                                    'txid': tx.get('txid', '')
                                })
                    
                    # Check if wallet is receiver (appears in outputs)
                    amount_received = 0
                    is_receiver = False
                    for out in tx.get('vout', []):
                        if out.get('scriptpubkey_address') == address:
                            amount_received += out.get('value', 0)
                            is_receiver = True
                    
                    # If receiver, create edges from all senders
                    if is_receiver:
                        for inp in tx.get('vin', []):
                            sender = inp.get('prevout', {}).get('scriptpubkey_address')
                            
                            if sender and sender != address:
                                edges_list.append({
                                    'source': sender,
                                    'target': address,
                                    'weight': amount_received,
                                    'timestamp': tx_time,
                                    'direction': 'incoming',
                                    'txid': tx.get('txid', '')
                                })
                break  # Success, exit retry loop
            
            elif r.status_code == 429:
                retries += 1
                wait_time = 5 * retries  # Exponential backoff
                print(f"      Rate limited! Waiting {wait_time}s... (retry {retries}/{max_retries})")
                time.sleep(wait_time)
            
            else:
                print(f"      HTTP {r.status_code} for {address}")
                break  # Non-retryable error
        
        except requests.exceptions.Timeout:
            retries += 1
            print(f"      Timeout! Retry {retries}/{max_retries}")
            time.sleep(2)
        except Exception as e:
            print(f"      Error fetching {address}: {e}")
            break
    
    return edges_list

In [3]:
def fetch_random_wallets(input_path, label, output_path, n_wallets=1000):
    """Fetch transactions for n random wallets, skipping already processed ones."""
    
    print(f"\n{'='*60}")
    print(f"Fetching {n_wallets} random {label.upper()} wallets")
    print(f"{'='*60}\n")
    
    # Load wallet data
    df = pd.read_csv(input_path, sep='\t', low_memory=False)
    df = df[df['address'].notna()].copy()
    
    # Load already processed wallets
    progress_file = output_path.replace('.csv', '_progress.txt')
    already_fetched = set()
    
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            already_fetched = set(line.strip() for line in f if line.strip())
    
    # Filter out already processed
    df = df[~df['address'].isin(already_fetched)].copy()
    print(f"Available wallets (not yet processed): {len(df)}")
    
    if len(df) == 0:
        print("All wallets already processed!")
        return
    
    # Sample random wallets
    n_to_fetch = min(n_wallets, len(df))
    df_sample = df.sample(n=n_to_fetch, random_state=random.randint(1, 10000))
    print(f"Selected {n_to_fetch} random wallets to fetch\n")
    
    # Prepare output file
    file_exists = os.path.exists(output_path)
    if not file_exists:
        pd.DataFrame(columns=['source', 'target', 'weight', 'timestamp', 'direction', 'txid', 'wallet_label']).to_csv(
            output_path, index=False, mode='w'
        )
    
    start_time = time.time()
    
    for idx, (i, row) in enumerate(df_sample.iterrows()):
        address = row['address']
        first_time = row.get('first_time', None)
        last_time = row.get('last_time', None)
        
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed if elapsed > 0 else 0
        remaining = (n_to_fetch - idx - 1) / rate if rate > 0 else 0
        
        print(f"[{idx+1}/{n_to_fetch}] {address[:25]}... | {elapsed/60:.1f}m elapsed | {remaining/60:.1f}m remaining")
        
        edges = fetch_wallet_transactions(address, first_time, last_time)
        
        if edges:
            print(f"      Found {len(edges)} edges")
            for edge in edges:
                edge['wallet_label'] = label
            pd.DataFrame(edges).to_csv(output_path, index=False, mode='a', header=False)
        
        # Mark as processed
        with open(progress_file, 'a') as f:
            f.write(f"{address}\n")
        
        time.sleep(1)
    
    print(f"\n{'='*60}")
    print(f"DONE! Fetched {n_to_fetch} {label} wallets in {(time.time()-start_time)/60:.1f} minutes")
    print(f"{'='*60}\n")

In [ ]:
# Fetch 1000 random benign wallets
bb_output = os.path.join(REAL_CATS_DATA_DIR, 'bb_transactions.csv')
fetch_random_wallets(PATH_BENIGN, 'benign', bb_output, n_wallets=1000)


Fetching 1000 random BENIGN wallets

Available wallets (not yet processed): 90164
Selected 1000 random wallets to fetch

[1/1000] 19E6qdJvejZDq5pGYoJ47MMjc... | 0.0m elapsed | 0.0m remaining
      Found 1 edges
[2/1000] 3MPFx2VVQFY8msV5LoQtmXdwM... | 0.0m elapsed | 11.3m remaining
[3/1000] bc1q42u72wmw62ftchmfrqfff... | 0.7m elapsed | 235.9m remaining
      Found 49 edges
[4/1000] 1MYqbAVnXTyHhhhtZb656dEcE... | 0.7m elapsed | 182.7m remaining
[5/1000] bc1qqauja4j64ge03jy8akqws... | 0.8m elapsed | 150.1m remaining
      Found 1 edges
[6/1000] 3N3dkwsCXg3rpeDVwywKg54e1... | 0.8m elapsed | 128.5m remaining
      Found 1 edges
[7/1000] 1JJGmL1skxw2JuydyGUP9sSYB... | 0.8m elapsed | 115.0m remaining
      Found 49 edges
[8/1000] 3QoJDDVSiCfBrtGb7MLLyR1w4... | 1.0m elapsed | 122.3m remaining
      Found 1 edges
[9/1000] 139PFHcSWdRvfqntaqXCEMEUZ... | 1.0m elapsed | 111.7m remaining
      Found 1 edges
[10/1000] 1FCdBNA7iJAvVBwkGQek1JqxQ... | 1.1m elapsed | 104.1m remaining
      Found 1 edge

In [ ]:
# Fetch 1000 random criminal wallets
cb_output = os.path.join(REAL_CATS_DATA_DIR, 'cb_transactions.csv')
fetch_random_wallets(PATH_CRIMINAL, 'criminal', cb_output, n_wallets=1000)